# OSQP Oefeningen

Installatie package:  `conda install -c conda-forge osqp`

### Voorbeeld 1
Minimaliseer $f(x) = \frac{1}{2}x^T P x + q^T x$ met beperking $l \leq Ax \leq u$ en met $$P =  \begin{bmatrix}
  4 & 1 \\
  1 & 2 \\
\end{bmatrix} , q = \begin{bmatrix}
  1 \\
  1 \\
\end{bmatrix} , A = \begin{bmatrix}
  1 & 1 \\
  -1 & 2 \\
  2 & 1 \\
\end{bmatrix}, l = \begin{bmatrix}
  1 \\
  0 \\
  2 \\
\end{bmatrix}, u = \begin{bmatrix}
  2 \\
  2 \\
  3 \\
\end{bmatrix}$$

In [1]:
import numpy as np
from scipy.sparse import csc_matrix #CSC staat voor compact sparse column. Dit is wat OSQP verwacht te krijgen
import osqp

# Matrices opstellen / ingeven
P = np.array([[4, 1], [1, 2]])
q = np.array([1, 1])
A = np.array([[1, 1], [-1, 2], [2, 1]])
l = np.array([1, 0, 2])
u = np.array([2, 2, 3])

# P en A moeten als sparse matrices worden doorgegeven
sparse_A = csc_matrix(A)
sparse_P = csc_matrix(P)

# OSQP initialisatie
prob = osqp.OSQP()

# setup
prob.setup(sparse_P, q, sparse_A, l, u)

# Oplossing
result = prob.solve()

print("Oplossing:")
print(result.x)


-----------------------------------------------------------------
           OSQP v0.6.3  -  Operator Splitting QP Solver
              (c) Bartolomeo Stellato,  Goran Banjac
        University of Oxford  -  Stanford University 2021
-----------------------------------------------------------------
problem:  variables n = 2, constraints m = 3
          nnz(P) + nnz(A) = 9
settings: linear system solver = qdldl,
          eps_abs = 1.0e-03, eps_rel = 1.0e-03,
          eps_prim_inf = 1.0e-04, eps_dual_inf = 1.0e-04,
          rho = 1.00e-01 (adaptive),
          sigma = 1.00e-06, alpha = 1.60, max_iter = 4000
          check_termination: on (interval 25),
          scaling: on, scaled_termination: off
          warm start: on, polish: off, time_limit: off

iter   objective    pri res    dua res    rho        time
   1  -2.6269e-01   2.93e+00Oplossing:
[0.80000997 0.4000221 ]
   1.20e+00   1.00e-01   3.02e-05s
  50   2.9601e+00   4.20e-05   6.20e-04   4.06e+00   8.49e-05s

status:        

In [2]:
result.x

array([0.80000997, 0.4000221 ])

### Voorbeeld 2

We willen $c^Tx$ minimaliseren, met randvoorwaarde $Gx \leq h$. We moeten nu zelf nog een beetje werk doen om dit om te zetten naar de formulering hierboven. Concreet is $$c =  \begin{bmatrix}
  1 \\
  -1 \\
\end{bmatrix} , G = \begin{bmatrix}
  1 & -1 \\
  -1 & -2 \\
  2 & 1 \\
\end{bmatrix}, h = \begin{bmatrix}
  -1 \\
  -2 \\
  4 \\
\end{bmatrix}$$

Als je goed kijkt zie je dat $h$ de bovengrens is. Er is geen ondergrens gegeven, dus we zullen een ondergrens maken door een array met $-\infty$ te vullen. $G$ is gewoon onze $A$, dus dat is makkelijk.

$c$ is het lineaire stuk van de te minimaliseren kostfunctie, dus $c = q$. Bij dit probleem is er geen kwadratisch stuk, dus voor de matrix $P$ gaan we een vierkante 0-matrix meegeven.

In [ ]:
import numpy as np
import osqp

# gegeven
#TODO

# Oefeningen

## Portfolio optimalisatie

*DISCLAIMER*: de onderstaande methode is in géén geval advies over hoe je geld beheert !!! 

De financiële markten zijn onderhevig aan veel onverwachte effecten (soms Black-Swan effecten genoemd, zoals de .dotcom bubble, de vastgoedcrisis in 2008, de covid aandelencrash en revibe in 2020-2021, ...) en wie beweert dat hij/zij de beurs kan voorspellen, dwaalt. In het beste geval kan je met kansen werken om bepaalde risico's te beperken.

Een financieel analyst wil haar aandelenportefeuille optimaliseren. Ze wil dus haar budget verdelen over verschillende aandelen, in de hoop om de verwachte return van die aandelen te maximaliseren.

*Doel*: maximaliseer de verwachte return $\sum_{i=1}^n w_i \cdot r_i$
Hierbij is $w_i$ het gewicht, wat we dus zo optimaal willen zoeken, en $r_i$ de verwachte return van aandeel $i$.

*Randvoorwaarden*
- De som van alle gewichtjes moet in totaal 1 zijn: $\sum_{i=1}^n w_i = 1$
- De analyst wil bovendien haar risico beperken. Hiervoor krijg je een covariance matrix tussen alle returns, die dus op rij $i$ en kolom $j$ aangeeft wat het verband is tussen die twee aandelen.

Gegeven data:

In [ ]:
import numpy as np

# aantal aandelen
num_stocks = 3

# Verwachte return (R)
np.random.seed(42)
expected_returns = np.random.uniform(0.05, 0.15, num_stocks) # hier ben je optimistisch over de beurssituatie

# correlatie matrix(Sigma)
correlation_matrix = np.array([[1.0, 0.5, -0.3],
                               [0.5, 1.0, 0.2],
                               [-0.3, 0.2, 1.0]])

# correlation matrix omzetten naar covariance matrix
volatility = np.random.uniform(0.1, 0.3, num_stocks) # in financiële sector wordt volatiliteit gebruikt als synoniem voor de standaardafwijking op aandelen
covariance_matrix = np.outer(volatility, volatility) * correlation_matrix

print("Verwachte Returns (R):")
print(expected_returns)

print("\nCovariance Matrix (Sigma):")
print(covariance_matrix)


Er is ook nog zoiets als een *risk-aversion parameter*: dit geeft aan in welke mate de gebruiker veel zin heeft voor risico's. Een hoge alpha wijst op weinig zin voor risico.

In [ ]:
alpha = 0.1 # risk aversion parameter

In [ ]:
# Matrices: nog veel zelf aan te vullen

# P is 2 keer het product van alpha met de covariance matrix

# denk goed na bij q !

# A: je kan starten met een A die bestaat uit een rij van 1'tjes voor elk aandeel

# lower en upper bound
l = np.array([1.0]) # de gewichtjes moeten sommeren tot 1
u = np.array([1.0]) # de gewichtjes moeten sommeren tot 1

# sparse matrix van maken
sparse_A = csc_matrix(A)
sparse_P = csc_matrix(P)

# Initialisatie
prob = osqp.OSQP()

# Setup
prob.setup(P=sparse_P, q=q, A=sparse_A, l=l, u=u)

# Oplossing
result = prob.solve()

print("Oplossing:")
print(result.x)


je merkt nu dat de oplossing negatieve getallen bevat, en getallen groter dan 1. Dat willen we niet, we zijn iets vergeten:

In [ ]:
# Matrices A, u, l

# Hoe kan je coderen dat elk item tussen 0 en 1 moet liggen ?
# Schrijf indien nodig op papier, en zet dan om naar een matrix
# Je kan stukken matrix bij elkaar gieten met np.vstack of gelijkaardige hulpmiddelen


In [ ]:
# Initialisatie
prob = osqp.OSQP()

# Setup
prob.setup(P=sparse_P, q=q, A=sparse_A, l=l, u=u)

# Oplossing
result = prob.solve()

print("Oplossing:")
print(result.x)

Laten we dit nu doen met een grotere dataset, dus met meer verschillende aandelen.

Installatie: `pip install yfinance`

We kunnen hiermee makkelijk aandelen bekijken, zoals de aandelen van Apple:

In [ ]:
import yfinance as yf
from datetime import datetime

today = datetime.now().strftime('%Y-%m-%d')
stock_data = yf.download('AAPL', start='2020-01-01', end=today)

In [ ]:
stock_data.tail()

We gaan werken met de volgende lijst van (Amerikaanse) aandelen.

In [ ]:
stocks_names = ['AAPL', 'GOOGL', 'MSFT', 'AMZN', 'META', 'TSLA', 'NVDA', 'V', 'JPM', 'JNJ',
                'PYPL', 'INTC', 'VZ', 'CSCO', 'DIS', 'GS', 'WMT', 'IBM', 'BA', 'GE',
                'GM', 'MCD', 'XOM', 'CVX', 'KO', 'PEP', 'PG', 'MRK', 'C', 'BAC',
                'WFC', 'INTU', 'AMGN', 'COST', 'GS', 'AAP', 'FDX', 'HON', 'UNH', 'MMM',
                'CAT', 'HD', 'IBM', 'WBA', 'PFE', 'ABT', 'CVS', 'TXN', 'QCOM', 'NKE']


Laad de data in door:
- een for lus te maken over alle stock names
- per stock name haal je data op van 3 weken geleden tot nu
- je voegt de stock name toe als kolom aan de tabel


Voeg alle dataframes samen door een lijst bij te houden van de dataframes, en op het einde een `pd.concat` toe te passen op de lijst van dataframes.

In [ ]:
import pandas as pd
from datetime import datetime, timedelta

today = datetime.now().strftime('%Y-%m-%d')
three_weeks_ago = (datetime.now() - timedelta(weeks=3)).strftime('%Y-%m-%d')

#todo

In [ ]:
all_stock_data.head()

De verwachte winst gaan we berekenen als simpelweg het verschil tussen het aandeel vandaag, en het aandeel 3 weken geleden. Dit is uiteraard een versimpeling, die in de realiteit geen enkele garantie geeft op succes. Pas de code aan met volgende snippet:

In [ ]:
first_close = all_stock_data[['Close']].iloc[0].mean()
    
stock_data['difference'] = stock_data['Close'] - first_close
stock_data['difference'] = stock_data['difference'] / first_close # procentueel

In [ ]:
#todo

Kijk nu naar de meest recente differences, per aandeel. Gebruik `.groupby`, `.last().values` om dit te bereiken. Steek uiteindelijk de resultaten in een vector.

In [ ]:
import numpy as np
# Laatste verschil nemen


# in vector steken



print("Diff Vector:")
print(expected_returns_big)


Nu berekenen we de covariantiematrix, die hebben we ook nodig voor ons model. Daarvoor heb je een `.pivot_table` nodig, eens je die hebt kan je `.cov` berekenen op die pivot tabel.

In [ ]:
# Pivoteer  DataFrame:'Close' prices als kolommen , stock names als rijen

# berekening covariance matrix


Nu kunnen we opnieuw OSQP gebruiken op dit grotere probleem:

In [ ]:
# Pas de OSQP code aan voor de grotere matrices 

In [ ]:
# Initialisatie
prob = osqp.OSQP()

# Setup
prob.setup(P=sparse_P, q=q, A=sparse_A, l=l, u=u)

# Oplossing
result = prob.solve()

print("Oplossing:")
print(result.x)

Welk aandeel werd het sterkst aanbevolen ?

In [ ]:
max_index = np.argmax(result.x)
stocks_names[max_index]

Giet dit nu uiteindelijk in een klasse, waar je de verwachte return vector en de variatiematrix kan meegeven. Op die manier heb je een wrapper gebouwd rond OSQP om specifiek het probleem van de aandelen op te lossen. Test je wrapper functie door de uitgbreidere lijst met aandelen er opnieuw door te sturen.

# Optimalisatie in een fabriek

We bekijken een probleem in een fabriek, waar een bedrijf zoveel mogelijk winst wil maken met 2 producten die ze kan maken. Er worden twee producten gemaakt: een product AquaSparkle, en een product BioBurst. Om deze producten te maken, heb je twee grondstoffen nodig (X en Y). Elk product heeft een bepaalde winst per verkochte eenheid, en de grondstoffenvoorraad is beperkt:
- Product AquaSparkle vereist 2 eenheden grondstof X, 1 eenheid grondstof Y
- Product BioBurst vereist 1 eenheid grondstof X, 3 eenheden grondstof Y
- Er zijn 100 grondstoffen X
- Er zijn 90 grondstoffen Y
De winst voor een verkocht AquaSparkle is 5 euro. De winst voor BioBurst is 8 euro. 

Bepaal het optimale productieproces: hoeveel AquaSparkles en hoeveel Biobursts moeten er worden gemaakt ?


### Code omzetten naar .py file

Zet je uiteindelijke code om in wrappercode. (basisversie)

Zorg in de wrappercode dat je bij initialisatie de volgende waardes kan aanpassen: (versie met parameters)
- de beperking op het aantal grondstoffen X
- de beperking op het aantal grondstoffen Y
- de winst van A
- de winst van B
- (de verhouding van grondstoffen voor de producten mag je constant houden en dus hardcoden)


In [ ]:
# TODO

# Optimalisatie van Macronutriënten in Maaltijden

Een topatleet streeft ernaar om zijn of haar dagelijkse voeding zorgvuldig af te stemmen op een specifieke verhouding van macronutriënten, namelijk proteïne, koolhydraten en vetten. De beschikbare ingrediënten voor deze maaltijden zijn broccoli ($B$), kalkoenfilet ($K$) en aardappelen ($A$), waarbij de hoeveelheden van elk ingrediënt in grammen moeten worden bepaald.

Er is de volgende nutritionele info beschikbaar over de ingrediënten:
- 100g Broccoli: 2.8g proteïne, 7g koolhydraten, 0.4g vetten
- 100g Kalkoenfilet: 29g proteïne, 0g koolhydraten, 7g vetten
- 100g Aardappelen:  2g proteïne, 17g koolhydraten, 0.1g vetten

Het doel is om de hoeveelheden van deze ingrediënten ($x_B$, $x_K$, $x_A$) te vinden die de afwijkingen van de gewenste doelhoeveelheden voor proteïne ($P$), koolhydraten ($C$) en vetten ($F$) minimaliseren. Dit optimalisatieprobleem kan worden gemodelleerd als een lineair programmeringsprobleem.

Heel concreet kun je starten met $P = 37, C = 41, F = 20$. Er zijn bovendien nog een aantal dingen om rekening mee te houden:
- De atleet eet in totaal exact 700g
- daarvan is de hoeveelheid broccoli tussen 200 en 300 gram
- de hoeveelheid kalkoen ligt tussen 100 en 200 gram
- de hoeveelheid aardappelen tussen 200 en 400 gram

### Stappenplan

1. Het moeilijkste is het omzetten van de zin *'Het doel is om de hoeveelheden van deze ingrediënten ($x_B$, $x_K$, $x_A$) te vinden die de afwijkingen van de gewenste doelhoeveelheden voor proteïne ($P$), koolhydraten ($C$) en vetten ($F$) minimaliseren.'* in een goede kostfunctie. Daar gaan we mee beginnen.

    - Hoe druk je uit dat de totale proteïnen dicht bij de waarde van P blijven ? Je moet hier een uitdrukking maken die $P$ (of gewoon 37) in verband brengt met de hoeveelheid proteïnen die je krijgt uit broccoli, kalkoen en aardappelen samen te brengen. Je wil in essentie dat het verschil tussen die twee dingen, klein is. Druk dit uit in een functie
    - Indien je in de vorige stap de absolute waarde hebt gebruikt om uit te drukken dat iets klein moet zijn, dan mag je die nu vervangen door de kwadraatfunctie. Ook die drukt een afstand uit, en is makkelijker om mee te (laten) rekenen.
    - Zoek een gelijkaardige uitdrukking voor $C$ en $F$.
    - De kostfunctie is nu de optelsom van deze 3 uitdrukkingen --> dit moeten we zo klein mogelijk maken.

Nu kan je www.wolframalpha.com gebruiken om deze uitdrukking te laten uitrekenen.  Je moet wel, als je iets aan Wolfram vraagt werken met variabelen x, y, z: iets anders kent Wolfram Alpha niet. Dus je vervangt $x_B$ door $x$, $x_K$ door $y$ etc. Je had ook in het begin meteen deze variabelen kunnen kiezen. Wolfram geeft een uitdrukking terug:

![image.png](attachment:image.png)

Deze uitdrukking moet je nu omzetten naar een matrixvorm:
De tabel hieronder toont hoe je zo'n uitdrukking terug kan omzetten in een matrix $P$, die je dus nodig hebt voor OSQP:

$$ P = \begin{bmatrix}
  a & d & e \\
  d & b & f \\
  e & f & c \\
\end{bmatrix} $$

Tabel
| Label | de coëfficient van |
|-------|-------------|
| a     |      $x_B^2$ * 2      |
| b     |      $x_K^2$ * 2    |
| c     |      $x_A^2$  * 2     |
| d     |      $x_B x_K$       |
| e     |      $x_B x_A$       |
| f     |      $x_K x_A$       |


Opgelet, de bovenste 3 coëfficiënten kunnen we niet rechtstreeks aflezen: we moeten wat we aflezen met 2 vermenigvuldigen en dat voor de letterwaarde invullen.

In [ ]:
# Bepaal hier de matrix P

Er komen nog waarden uit deze uitdrukking (de lineaire stukken). Die horen tot de vector $q$:

In [ ]:
q = np.array([-797, -2626, -1546])

*oplossing*
$$
\begin{align*}
\text{Randvoorwaarden} & \\
&x_B, x_K, x_A \geq 0 \quad \text{(hoeveelheden kunnen niet negatief zijn!)} \\
&2 \leq x_B \leq 3 \qquad \text{(ik eet zeker tussen 200 en 300g broccoli)}  \\
&1 \leq x_K \leq 2 \\
&2 \leq x_A \leq 4 \\
& x_B + x_K + x_A = 7 \qquad \text{(ik eet in totaal 700g)} \\
\end{align*}
$$

Zet deze nu naar code en laat OSQP het werk doen.